In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import hdbscan
import umap

In [ ]:
overall_df = pd.read_csv("2015_to_2023_full_preprocessed_data_set.csv")

In [ ]:
cluster_cols = ["Age9", "Gend3", "Eth7", "Disab3", "NSSEC5", "Educ6"]

overall_df[cluster_cols].isna().mean().sort_values()

In [ ]:
for col in cluster_cols:
    print(overall_df[col].value_counts(dropna=False))

In [ ]:
cluster_df = overall_df[cluster_cols + ["year"]].dropna().copy()

In [ ]:
X = pd.get_dummies(cluster_df[cluster_cols], drop_first=False)

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=1000, min_samples=25, prediction_data=True)

labels = clusterer.fit_predict(X) # type: ignore

cluster_df["cluster"] = labels

cluster_df["cluster"].value_counts()

In [ ]:
noise_points = (cluster_df["cluster"] == -1).sum()

cluster_sizes = (cluster_df.loc[cluster_df["cluster"] != -1, "cluster"].value_counts())

cluster_summary = pd.DataFrame({"Metric": ["Observations", "Clusters", "Noise points", "Largest cluster", "Smallest cluster", "Mean cluster size"],
                        "Value": [
                            len(cluster_df),
                            cluster_df["cluster"].nunique() - (1 if -1 in cluster_df["cluster"].unique() else 0),
                            noise_points,
                            cluster_sizes.max(),
                            cluster_sizes.min(),
                            round(cluster_sizes.mean(), 1)]})

print(cluster_summary)

In [ ]:
noise_pct = ((cluster_df["cluster"] == -1).mean() * 100)

print(f"Noise percentage: {noise_pct:.2f}%")

In [ ]:
cluster_years = pd.crosstab(cluster_df["cluster"], cluster_df["year"], normalize="columns")

print(cluster_years.round(3))

In [ ]:
for col in cluster_cols:

    print(f"\n{'='*60}")
    print(col)

    print(pd.crosstab( cluster_df["cluster"], cluster_df[col], normalize="index").round(3))

In [ ]:
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42)

embedding = reducer.fit_transform(X)

plt.figure(figsize=(10, 8))

plt.scatter(embedding[:, 0], embedding[:, 1], c=cluster_df["cluster"], s=5, cmap="tab20")  # type: ignore

plt.title("HDBSCAN Clusters")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")

plt.show()